# 이자비용과 영업이익이 함께 늘어난 업종에서는 무슨 일이 있었을까?

이전 분석에서 제시한 숫자를 같은 자료와 단위로 다시 계산한다. 처음 읽는 사람은 아래의 업종 수 비교부터 보면 이 연구가 설명하려는 차이를 볼 수 있다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# 대화 간 결과 대조용 검산
2026-11-03 사용자가 요청한 전체 분석 정리를 위해 참조 대화의 수치를 현재 로컬 입력으로 대조한다. 새로운 가설 검정이나 최종 주 모형 변경이 아니다. 참조 대화 원 실행 코드는 확보되지 않았으므로 이 재계산이 원 코드를 검증했다는 뜻은 아니다.

참조 주장 1: 기존 E(단기차입/자산) 모형의 평균잔액 비용 대용률 계수 10%p당 약 -0.002, p=.997. 주장 2: Q/S 분리 모형의 영업이익/전기자산 QP 10%p당 약 +4.159, p=.026. 둘 다 업종·연도FE, Post22–23, 2021 별도, 사전18–20, L기간항 포함이라고 설명되어 있다. 동일하게 재구성하여 비교한다. code·year 키로 병합, 입력 해시, 전체 40업종 유지. 계수·SE는 CR1 업종 군집, t39. 추가 bootstrap이나 다중검정은 하지 않았고 원 주장과의 수치 대조 용도로만 보고한다.

또한 이자비용 금액과 비용률의 38/40은 다른 지표임을 확인하며, 평균(4.12→5.19)과 중앙값(4.14→5.16)의 차이도 분리한다. inputs 원본 복사본을 포함, Python3.12 및 requirements.txt, 노트북 폴더에서 실행.


In [1]:
# 비교 지표와 단위 맞추기
from pathlib import Path
import json,hashlib
import numpy as np,pandas as pd,statsmodels.api as sm
from scipy import stats
from IPython.display import display
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
def save(x, n):
    x.to_csv(BASE / 'tables' / f'{n}.csv', index=False, encoding='utf-8-sig')
for x in json.loads((BASE/'inputs/manifest.json').read_text(encoding='utf-8')):
 assert hashlib.sha256((BASE/'inputs'/x['file']).read_bytes()).hexdigest()==x['sha256']
t = pd.read_csv(BASE / 'inputs/mechanism_panel.csv')
o = pd.read_csv(BASE / 'inputs/original_panel.csv')
t=t.merge(o[['code','year','EP','LP','ET','LT']],on=['code','year'],suffixes=('','_orig'),validate='one_to_one')
t['profit_asset']=100*t['영업이익']/t.lagA
def design(d,terms):
    X=pd.concat([pd.DataFrame({'const':np.ones(len(d))},index=d.index),d[terms],pd.get_dummies(d.code,prefix='i',drop_first=True,dtype=float),pd.get_dummies(d.year,prefix='y',drop_first=True,dtype=float)],axis=1).astype(float)
    assert np.linalg.matrix_rank(X)==X.shape[1]
    return X
fits = {}
rows = []
def fit(label, d, terms, y='Y'):
    X = design(d, terms)
    model = sm.OLS(d[y].astype(float), X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d.code, 'use_correction': True, 'df_correction': True}, use_t=True)
    G = d.code.nunique()
    q = stats.t.ppf(0.975, G - 1)
    for term in terms:
        b = float(res.params[term])
        se = float(res.bse[term])
        rows.append(dict(model=label, outcome=y, term=term, beta=b, se=se, lo=b - q * se, hi=b + q * se, p=float(2 * stats.t.sf(abs(b / se), G - 1)), N=len(d), G=G))
    fits[label] = (d.copy(), terms, model, res)
    return res

# 비용률과 수익성 추정치 대조
fit('reference_E_cost',t,['EP','LP','ET','LT'],'cost')
fit('reference_QS_profit_asset',t,['QP','SP','LP','QT','ST','LT'],'profit_asset')
r = pd.DataFrame(rows)
r['beta_per10pp'] = 10 * r.beta
save(r, 'reference_regression_check')
display(r)

# 업종별 증가·감소 집계
a=t[t.year.between(2018,2020)].groupby('code')[['영업이익','이자비용','icr','cost']].mean()
b=t[t.year.isin([2022,2023])].groupby('code')[['영업이익','이자비용','icr','cost']].mean()
u=b-a
counts={'interest_up':int((u['이자비용']>0).sum()),'profit_up':int((u['영업이익']>0).sum()),'icr_down':int((u.icr<0).sum()),'cost_up':int((u.cost>0).sum()),'interest_up_profit_down':int(((u['이자비용']>0)&(u['영업이익']<0)).sum()),'interest_up_profit_down_icr_down':int(((u['이자비용']>0)&(u['영업이익']<0)&(u.icr<0)).sum()),'both_up_icr_up':int(((u['이자비용']>0)&(u['영업이익']>0)&(u.icr>0)).sum()),'both_up_icr_down':int(((u['이자비용']>0)&(u['영업이익']>0)&(u.icr<0)).sum()),'cost_pre_mean':a.cost.mean(),'cost_post_mean':b.cost.mean(),'cost_pre_median':a.cost.median(),'cost_post_median':b.cost.median()}
save(pd.DataFrame([counts]), 'reference_counts_check')
display(pd.DataFrame([counts]))

# 업종 사례 저장
names=t.drop_duplicates('code').set_index('code').industry
case=pd.DataFrame({'industry':names,'profit_change_pct':100*u['영업이익']/a['영업이익'],'interest_change_pct':100*u['이자비용']/a['이자비용'],'official_icr_pre':a.icr,'official_icr_post':b.icr,'cost_pre':a.cost,'cost_post':b.cost}).reset_index()
save(case,'reference_cases_check')
print('Cross-chat verification complete; no new hypothesis/model selection performed.')


                       model       outcome term  ...    N   G  beta_per10pp
0           reference_E_cost          cost   EP  ...  240  40     -0.001866
1           reference_E_cost          cost   LP  ...  240  40      0.034901
2           reference_E_cost          cost   ET  ...  240  40     -0.247748
3           reference_E_cost          cost   LT  ...  240  40     -0.074125
4  reference_QS_profit_asset  profit_asset   QP  ...  240  40      4.158899
5  reference_QS_profit_asset  profit_asset   SP  ...  240  40      0.045485
6  reference_QS_profit_asset  profit_asset   LP  ...  240  40      0.547269
7  reference_QS_profit_asset  profit_asset   QT  ...  240  40      0.780348
8  reference_QS_profit_asset  profit_asset   ST  ...  240  40     -0.486550
9  reference_QS_profit_asset  profit_asset   LT  ...  240  40      0.298019

[10 rows x 11 columns]
   interest_up  profit_up  ...  cost_pre_median  cost_post_median
0           38         24  ...         4.144271          5.159146

[1 rows

### 출력에서 읽을 수 있는 것

이자비용이 늘어난 곳은 38개다. 이 중 영업이익이 줄어든 14곳에서는 모두 ICR이 하락했다. 두 금액이 모두 늘어난 24곳에서는 ICR이 15곳에서 오르고 9곳에서 내렸다.

- 이전 대화의 숫자를 같은 단위로 다시 비교할 수 있도록 입력과 분모를 맞췄다.
- 사전 단기차입 의존도 10%포인트당 비용률 변화 계수는 −0.001866(p=0.997)이었다. 총차입 대용지표와 영업이익/전기자산 관계는 +4.159(p=0.026)였다. 두 값은 결과변수부터 다르다.
- 이자비용이 증가하고 영업이익이 감소한 14개 업종 모두 ICR이 하락했다. 둘 다 증가한 24개 업종에서는 15개가 상승하고 9개가 하락했다.
- 개별 사례를 다시 볼 수 있도록 업종별 증감 자료를 저장했다.